In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import hdbscan
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

COLLECTION_QWEN = "SEMImages_qwen"
COLLECTION_DINO = "SEMImages_dino"

N_PERMUTATIONS = 5000
RANDOM_SEED = 0

In [ ]:
def fetch_points(collection_name):
    points, next = client.scroll(collection_name=collection_name, limit=1000, with_payload=True, with_vectors=True)
    vectors, payloads, ids = [], [], []

    for p in points:
        vectors.append(p.vector)
        payloads.append(p.payload)
        ids.append(p.id)

    X = np.array(vectors)
    X_normalized = normalize(X, norm="l2", axis=1)
    return X_normalized, payloads, ids

Qdrant's collections are configured to use cosine similarity (models are trained so that direction, not magnitude, carries the semantic meaning). If Qdrant is retrieving by cosine similarity, any offline analysis you do should measure similarity the same way — otherwise your clustering/silhouette/distance results could disagree with what your actual retrieval system considers "similar," which would make the analysis misleading for your actual use case.  
Most standard tools operate on Euclidean distance, not cosine similarity $\rightarrow$ different metrics on raw vectors: two vectors can be very close in cosine similarity (same direction) but far apart in Euclidean distance if one has a much larger magnitude, or vice versa.  
$\Rightarrow$ for unit-length vectors, Euclidean distance and cosine distance are monotonically related. So once every vector has length 1, "closer in Euclidean distance" and "closer in cosine similarity" always agree.

In [ ]:
X_norm_QWEN, payload_QWEN, ids_QWEN = fetch_points(COLLECTION_QWEN)

In [ ]:
X_norm_DINO, payload_DINO, ids_DINO = fetch_points(COLLECTION_DINO)

In [ ]:
inertias, sil_scores = [], []
k_range = range(2, 15)

# KMEANS ON QWEN
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_norm_QWEN)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_norm_QWEN, km.labels_))

In [ ]:
# --- PLOTTING ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Inertia (Elbow Method) Plot
ax1.plot(k_range, inertias, marker='o', color='b', linestyle='-')
ax1.set_title('Elbow Method (Inertia)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.grid(True)

# 2. Silhouette Score Plot
ax2.plot(k_range, sil_scores, marker='o', color='g', linestyle='-')
ax2.set_title('Silhouette Score Analysis')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=50, random_state=42)
pca.fit(X_norm_QWEN)

print(f"Variance explained by 50 components: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
pca_full = PCA(random_state=42).fit(X_norm_QWEN)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(18,9))
plt.plot(cumvar)
plt.axhline(0.90, color="r", linestyle="--")
plt.axhline(0.80, color="g", linestyle="--")
plt.xlabel("Number of components")
plt.ylabel("Cumulative variance explained")
plt.show()

n_components = np.argmax(cumvar >= 0.90) + 1
print(f"{n_components} components explain 90% of variance")
n_components = np.argmax(cumvar >= 0.80) + 1
print(f"{n_components} components explain 80% of variance")
n_components = np.argmax(cumvar >= 0.70) + 1
print(f"{n_components} components explain 70% of variance")

In [ ]:
X_pca = PCA(n_components=n_components, random_state=42).fit_transform(X_norm_QWEN)

inertias_pca, sil_scores_pca = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_pca)
    inertias_pca.append(km.inertia_)
    sil_scores_pca.append(silhouette_score(X_pca, km.labels_))

In [ ]:
# --- PLOTTING ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Inertia (Elbow Method) Plot
ax1.plot(k_range, inertias_pca, marker='o', color='b', linestyle='-')
ax1.set_title('Elbow Method (Inertia)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.grid(True)

# 2. Silhouette Score Plot
ax2.plot(k_range, sil_scores_pca, marker='o', color='g', linestyle='-')
ax2.set_title('Silhouette Score Analysis')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')
hdb_labels = clusterer.fit_predict(X_pca)  # use the same PCA-reduced, normalized data

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print(f"Clusters found: {n_clusters}, noise points: {n_noise}/{len(hdb_labels)}")

# Payload Analysis

In [ ]:
df = pd.DataFrame(payload_QWEN)

SETTINGS_KEYS = [
    "AP_WD",
    "AP_ALIAS_WD",
    "AP_FREE_WD",
    "AP_BEAM_TIME",
    "AP_STAGE_LOW_Z",
    "AP_STAGE_CENTRE_X",
    "AP_C3",
    "AP_EHT_CAL_30KV",
    "AP_EHT_CAL_10KV",
    "AP_EHT_CAL_100",
    "AP_EHT_CAL_24KV",
    "AP_MINMAG",
]

In [ ]:
df.head()

In [ ]:
df.meta[0]['AP_WD']

In [ ]:
# Flatten the 'meta' column of dicts into its own dataframe of columns
meta_df = pd.json_normalize(df["meta"])

In [ ]:
meta_df.head()

In [ ]:
# Join back onto the original df (keep id/content around for reference if useful)
data = pd.concat([df.drop(columns=["meta"]), meta_df], axis=1)

print(data[SETTINGS_KEYS].head())

In [ ]:
data.head()

In [ ]:
for field in SETTINGS_KEYS:
    n_unique = data[field].nunique(dropna=True)
    n_missing = data[field].isna().sum()
    print(f"{field}: {n_unique} unique values, {n_missing} missing")

In [ ]:
BINARY_FIELDS = ["AP_STAGE_LOW_Z", "AP_STAGE_CENTRE_X", "AP_EHT_CAL_30KV",
                 "AP_EHT_CAL_10KV", "AP_EHT_CAL_100", "AP_EHT_CAL_24KV"]
CONTINUOUS_FIELDS = ["AP_WD", "AP_ALIAS_WD", "AP_FREE_WD", "AP_BEAM_TIME", "AP_C3", "AP_MINMAG"]

results = {}

for field in BINARY_FIELDS:
    valid = data[field].notna()
    vals = data.loc[valid, field]
    if vals.nunique() < 2:
        continue
    score = silhouette_score(X_norm_DINO[valid.values], vals.astype(str))
    results[field] = score
    print(f"{field} (binary): silhouette = {score:.4f}")

for field in CONTINUOUS_FIELDS:
    valid = data[field].notna()
    vals = data.loc[valid, field]
    try:
        bins = pd.qcut(vals, q=4, duplicates="drop")
    except ValueError:
        bins = vals
    if bins.nunique() < 2:
        continue
    score = silhouette_score(X_norm_DINO[valid.values], bins.astype(str))
    results[field] = score
    print(f"{field} (binned): silhouette = {score:.4f}  ({bins.nunique()} groups)")

ranked = sorted(results.items(), key=lambda x: -x[1])
print("\nRanked by silhouette score:")
for field, score in ranked:
    print(f"  {field}: {score:.4f}")

# Pairwise Cosine Distance Matrix for each embedding space

In [ ]:
# Cosine similarity = dot product of unit vectors
sim_qwen = X_norm_QWEN @ X_norm_QWEN.T
sim_dino = X_norm_DINO @ X_norm_DINO.T

# Convert to distance if you prefer thinking in those terms (1 - similarity)
dist_qwen = 1 - sim_qwen
dist_dino = 1 - sim_dino

In [ ]:
Z = linkage(dist_qwen[np.triu_indices(len(dist_qwen), k=1)], method="average")
order = leaves_list(Z)

plt.figure(figsize=(8, 8))
plt.imshow(sim_qwen[order][:, order], cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.title("QWEN cosine similarity (hierarchically ordered)")
plt.show()

In [ ]:
valid = data["AP_WD"].notna()
order_by_wd = data.loc[valid, "AP_WD"].argsort().values

plt.figure(figsize=(8, 8))
plt.imshow(sim_qwen[valid.values][order_by_wd][:, valid.values][:, order_by_wd], cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.title("QWEN similarity, sorted by AP_WD")
plt.show()

In [ ]:
Z = linkage(dist_qwen[np.triu_indices(len(dist_qwen), k=1)], method="average")

# Try a few distance thresholds and see how many clusters + sizes result
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    labels_h = fcluster(Z, t=t, criterion="distance")
    sizes = np.bincount(labels_h)
    n_clusters = len(sizes) - 1  # bincount includes a 0 bin
    print(f"t={t}: {n_clusters} clusters, sizes: {sorted(sizes[sizes>0], reverse=True)[:10]}")

# Dissimilar Points

In [ ]:
# Find the two most dissimilar points in QWEN space
D = squareform(pdist(X_norm_QWEN, metric="cosine"))
i, j = np.unravel_index(np.argmax(D), D.shape)
print(f"Most dissimilar pair: index {i} and {j}, distance {D[i,j]:.3f}")

In [ ]:
folder = "data/subset_test_1000/crops"

row = data.iloc[i]
files = [filename for filename in os.listdir(folder) if filename.startswith(f"{row["SEMImageID"]}")]

prova = np.load(f"{folder}/{files[0]}")

plt.imshow(prova, cmap='gray')
plt.show()

In [ ]:
folder = "data/subset_test_1000/crops"

row = data.iloc[j]
files = [filename for filename in os.listdir(folder) if filename.startswith(f"{row["SEMImageID"]}")]

prova = np.load(f"{folder}/{files[0]}")

plt.imshow(prova, cmap='gray')
plt.show()

In [ ]:
def top_k_dissimilar(D, query_idx, k=10):
    row = D[query_idx].copy()
    furthest_idx = np.argsort(row)[-k:]
    distances = row[furthest_idx]
    return furthest_idx, distances

furthest_idx, distances = top_k_dissimilar(D, query_idx=152, k=10)
for idx, dist in zip(furthest_idx, distances):
    print(f"index {idx}, SEMImageID {data.iloc[idx]['SEMImageID']}, distance {dist:.4f}")

In [ ]:
def load_crop(sem_id, folder="data/subset_test_1000/crops"):
    files = [f for f in os.listdir(folder) if f.startswith(str(sem_id))]
    return np.load(os.path.join(folder, files[0]))

def show_top_k(query_idx, furthest_idx, distances, data, folder="data/subset_test_1000/crops"):
    n = len(furthest_idx) + 1
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))

    query_id = data.iloc[query_idx]["SEMImageID"]
    axes[0].imshow(load_crop(query_id, folder), cmap="gray")
    axes[0].set_title(f"QUERY\n{query_id}")
    axes[0].axis("off")

    for ax, idx, dist in zip(axes[1:], furthest_idx, distances):
        sem_id = data.iloc[idx]["SEMImageID"]
        ax.imshow(load_crop(sem_id, folder), cmap="gray")
        ax.set_title(f"{sem_id}\nd={dist:.3f}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_top_k(152, furthest_idx, distances, data)

# Similar Points

In [ ]:
# Find the two most similar points in QWEN space
D = squareform(pdist(X_norm_QWEN, metric="cosine"))
np.fill_diagonal(D, np.inf)
i, j = np.unravel_index(np.argmin(D), D.shape)
print(f"Most similar pair: index {i} and {j}, distance {D[i,j]:.3f}")

In [ ]:
folder = "data/subset_test_1000/crops"

row = data.iloc[i]
files = [filename for filename in os.listdir(folder) if filename.startswith(f"{row["SEMImageID"]}")]

prova = np.load(f"{folder}/{files[0]}")

plt.imshow(prova, cmap='gray')
plt.show()

In [ ]:
folder = "data/subset_test_1000/crops"

row = data.iloc[j]
files = [filename for filename in os.listdir(folder) if filename.startswith(f"{row["SEMImageID"]}")]

prova = np.load(f"{folder}/{files[0]}")

plt.imshow(prova, cmap='gray')
plt.show()

In [ ]:
def top_k_similar(D, query_idx, k=10):
    row = D[query_idx].copy()
    row[query_idx] = np.inf  # exclude self-match
    nearest_idx = np.argsort(row)[:k]
    distances = row[nearest_idx]
    return nearest_idx, distances

nearest_idx, distances = top_k_similar(D, query_idx=i, k=10)
for idx, dist in zip(nearest_idx, distances):
    print(f"index {idx}, SEMImageID {data.iloc[idx]['SEMImageID']}, distance {dist:.4f}")

In [ ]:
def load_crop(sem_id, folder="data/subset_test_1000/crops"):
    files = [f for f in os.listdir(folder) if f.startswith(str(sem_id))]
    return np.load(os.path.join(folder, files[0]))

def show_top_k(query_idx, nearest_idx, distances, data, folder="data/subset_test_1000/crops"):
    n = len(nearest_idx) + 1
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))

    query_id = data.iloc[query_idx]["SEMImageID"]
    axes[0].imshow(load_crop(query_id, folder), cmap="gray")
    axes[0].set_title(f"QUERY\n{query_id}")
    axes[0].axis("off")

    for ax, idx, dist in zip(axes[1:], nearest_idx, distances):
        sem_id = data.iloc[idx]["SEMImageID"]
        ax.imshow(load_crop(sem_id, folder), cmap="gray")
        ax.set_title(f"{sem_id}\nd={dist:.3f}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_top_k(i, nearest_idx, distances, data)

In [ ]:
query_vector = X_norm_QWEN[i]  # or any embedding vector

results = client.query_points(
    collection_name="SEMImages_qwen",
    query=query_vector.tolist(),
    limit=10,
)
for point in results.points:
    print(point.payload['content'])
    print(point.score)  # score is similarity (higher=closer) since Qdrant collection is cosine-configured

In [ ]:
# Find the two most similar points in QWEN space
D = squareform(pdist(X_norm_DINO, metric="cosine"))
np.fill_diagonal(D, np.inf)
i, j = np.unravel_index(np.argmin(D), D.shape)
print(f"Most similar pair: index {i} and {j}, distance {D[i,j]:.3f}")

def top_k_similar(D, query_idx, k=10):
    row = D[query_idx].copy()
    row[query_idx] = np.inf  # exclude self-match
    nearest_idx = np.argsort(row)[:k]
    distances = row[nearest_idx]
    return nearest_idx, distances

nearest_idx, distances = top_k_similar(D, query_idx=140, k=10)
for idx, dist in zip(nearest_idx, distances):
    print(f"index {idx}, SEMImageID {data.iloc[idx]['SEMImageID']}, distance {dist:.4f}")

def load_crop(sem_id, folder="data/subset_test_1000/crops"):
    files = [f for f in os.listdir(folder) if f.startswith(str(sem_id))]
    return np.load(os.path.join(folder, files[0]))

def show_top_k(query_idx, nearest_idx, distances, data, folder="data/subset_test_1000/crops"):
    n = len(nearest_idx) + 1
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))

    query_id = data.iloc[query_idx]["SEMImageID"]
    axes[0].imshow(load_crop(query_id, folder), cmap="gray")
    axes[0].set_title(f"QUERY\n{query_id}")
    axes[0].axis("off")

    for ax, idx, dist in zip(axes[1:], nearest_idx, distances):
        sem_id = data.iloc[idx]["SEMImageID"]
        ax.imshow(load_crop(sem_id, folder), cmap="gray")
        ax.set_title(f"{sem_id}\nd={dist:.3f}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_top_k(140, nearest_idx, distances, data)

# Representational Similarity Analysis

Core Representational Similarity Analysis between DINO and QWEN embedding spaces.

Steps:
1. Fetch embeddings for shared IDs from both Qdrant collections, aligned by ID.
2. Build cosine-distance RDMs for each space.
3. Spearman-correlate the two RDMs (upper triangle, excluding diagonal).
4. Permutation test (label shuffling) for a proper p-value.

In [ ]:
def fetch_vectors_by_ids(client: QdrantClient, collection: str, ids: list[str]) -> np.ndarray:
    """Retrieve vectors for a specific list of IDs, in that exact order."""
    points = client.retrieve(
        collection_name=collection,
        ids=ids,
        with_vectors=True,
        with_payload=False,
    )
    by_id = {p.id: np.array(p.vector) for p in points}
    missing = [i for i in ids if i not in by_id]
    if missing:
        raise ValueError(f"{len(missing)} IDs missing from {collection}, e.g. {missing[:5]}")
    return np.stack([by_id[i] for i in ids])

In [ ]:
def rdm(embeddings: np.ndarray) -> np.ndarray:
    """Condensed cosine-distance RDM (upper triangle, as a 1D vector)."""
    return pdist(embeddings, metric="cosine")

In [ ]:
def permutation_test(dino_emb: np.ndarray, qwen_emb: np.ndarray, observed_rho: float,
                      n_perm: int, seed: int) -> float:
    """
    Null distribution: shuffle item labels on one space's embeddings (jointly
    permuting rows AND columns of its RDM), recompute correlation each time.
    """
    rng = np.random.default_rng(seed)
    n_items = qwen_emb.shape[0]
    qwen_square = squareform(rdm(qwen_emb))

    null_rhos = np.empty(n_perm)
    dino_condensed = rdm(dino_emb)
    for i in range(n_perm):
        perm = rng.permutation(n_items)
        shuffled = qwen_square[np.ix_(perm, perm)]
        shuffled_condensed = squareform(shuffled, checks=False)
        null_rhos[i], _ = spearmanr(dino_condensed, shuffled_condensed)

    # two-sided p-value
    p_value = (np.sum(np.abs(null_rhos) >= abs(observed_rho)) + 1) / (n_perm + 1)
    return p_value, null_rhos

In [ ]:
shared_ids = sorted(set(ids_DINO) & set(ids_QWEN))
print(f"DINO: {len(ids_DINO)} | QWEN: {len(ids_QWEN)} | shared: {len(shared_ids)}")

In [ ]:
dino_by_id = dict(zip(ids_DINO, X_norm_DINO))
qwen_by_id = dict(zip(ids_QWEN, X_norm_QWEN))

dino_emb = np.stack([dino_by_id[i] for i in shared_ids])
qwen_emb = np.stack([qwen_by_id[i] for i in shared_ids])
print(f"DINO shape: {dino_emb.shape}, QWEN shape: {qwen_emb.shape}")

In [ ]:
dino_rdm = rdm(dino_emb)
qwen_rdm = rdm(qwen_emb)

rho, _ = spearmanr(dino_rdm, qwen_rdm)
print(f"\nObserved Spearman RSA correlation: {rho:.4f}")

print(f"Running permutation test ({N_PERMUTATIONS} permutations)...")
p_value, null_rhos = permutation_test(dino_emb, qwen_emb, rho, N_PERMUTATIONS, RANDOM_SEED)
print(f"Permutation p-value: {p_value:.5f}")
print(f"Null distribution: mean={null_rhos.mean():.4f}, std={null_rhos.std():.4f}")

print("\n--- RSA summary ---")
print(f"N items:              {len(shared_ids)}")
print(f"Observed Spearman rho: {rho:.4f}")
print(f"Permutation p-value:   {p_value:.5f}")
print(f"Null mean / std:       {null_rhos.mean():.4f} / {null_rhos.std():.4f}")
print(f"Null min / max:        {null_rhos.min():.4f} / {null_rhos.max():.4f}")

In [ ]:
from qdrant_client import models

In [ ]:
point = client.scroll(
    collection_name=COLLECTION_QWEN,
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="id",
                match=models.MatchValue(value="111"),
            )
        ]
    ),
    with_payload=["meta.thumb_b64"],
    limit=1,
)[0]

In [ ]:
point[0].payload['meta']['thumb_b64']